In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

import time
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm import tqdm
import copy
import pandas as pd




Parameters

In [45]:
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 200
NUM_WORKERS = 2
IMAGE_HEIGHT = 318  
IMAGE_WIDTH = 318   
IN_CHANNELS = 20
PIN_MEMORY = True
LOAD_MODEL = False
TRAIN_DIR = "/home/st-juho/code_testing/split_dataset/train"
VAL_DIR = "/home/st-juho/code_testing/split_dataset/val"

Train and Val Shenanigans if Needed

In [46]:
# import os, shutil
# import numpy as np
# from PIL import Image
# from tqdm import tqdm

# SRC = "/home/st-juho/code_testing/split_dataset"
# DST = "/home/st-juho/code_testing/split_dataset_filtered"
# LRS = ["mask", "coh_vh_diff", "coh_vv_diff", "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood", 
#        "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary", "dem", "land_use", 
#        "buildings", "inlets", "outlets", "open_drains", "gravity_mains"]

# for splt in ["train", "val"]:
#     s_path, d_path = os.path.join(SRC, splt), os.path.join(DST, splt)
#     os.makedirs(d_path, exist_ok=True)
#     masks = [f for f in os.listdir(s_path) if f.endswith("_mask.png")]
    
#     print(f"Filtering {splt}...")
#     count = 0
#     for m in tqdm(masks):
#         if np.max(np.array(Image.open(os.path.join(s_path, m)))) > 0:
#             base = m.replace("_mask.png", "")
#             for l in LRS:
#                 f = f"{base}_{l}.png"
#                 if os.path.exists(os.path.join(s_path, f)):
#                     shutil.copy2(os.path.join(s_path, f), os.path.join(d_path, f))
#             count += 1
            
#     # Emergency move if val is empty
#     if splt == "val" and count == 0:
#         t_path = os.path.join(DST, "train")
#         tm = [f for f in os.listdir(t_path) if f.endswith("_mask.png")][0]
#         base = tm.replace("_mask.png", "")
#         for l in LRS:
#             shutil.move(os.path.join(t_path, f"{base}_{l}.png"), os.path.join(d_path, f"{base}_{l}.png"))
#         print(f"Moved {base} to val (val was empty)")

# print(f"Done. Filtered data at: {DST}")

Code

U-Net definitions

In [47]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)
    
class UNET(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, out_channels=1, features=[64, 128, 256, 512]):
        super(UNET, self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Down part of UNET
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Up part of UNET
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(
                    feature*2, feature, kernel_size=2, stride=2,
                )
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            #Check if sizes are same, change method if needed / wanted
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.final_conv(x)
    

def test():
    x = torch.randn((3,1,160,160))
    model = UNET(in_channels=1, out_channels=1)
    preds = model(x)
    print(preds.shape)
    print(x.shape)
    assert preds.shape == x.shape

if __name__ == "__main__":
    test()

torch.Size([3, 1, 160, 160])
torch.Size([3, 1, 160, 160])


Importing functions

In [48]:
class InSARDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Collects all files ending in _mask.png
        self.mask_files = sorted([f for f in os.listdir(root_dir) if f.endswith("_mask.png")])

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, index):
        mask_filename = self.mask_files[index]
        
        # Strips '_mask.png' to get the base ID: 'tile_y000000_x000636'
        base_tile = mask_filename.rsplit('_mask.png', 1)[0]
        
        # The 17 input feature suffixes
        suffixes = [
            "coh_vh_diff", "coh_vv_diff", 
            "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood",
            "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary",
            "dem", "land_use", "buildings",
            "inlets", "outlets", "open_drains", "gravity_mains", "roads", "slope", "streamflow_direction"
        ]
        
        layers = []
        for s in suffixes:
            file_path = os.path.join(self.root_dir, f"{base_tile}_{s}.png")
            
            # Load each layer as grayscale
            img = np.array(Image.open(file_path).convert("L"), dtype=np.float32)
            layers.append(img)

        # Stack into [H, W, 17] for the augmentation pipeline
        image = np.stack(layers, axis=-1)
        
        # Load the Ground Truth Mask
        mask_path = os.path.join(self.root_dir, mask_filename)
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)

        # Convert mask to binary (0.0 or 1.0)
        mask[mask > 0] = 1.0 

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations['image']
            mask = augmentations['mask']

        return image, mask

Utility functions

In [49]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])

def get_loaders(
    train_dir,
    train_maskdir,
    val_dir,
    val_maskdir,
    batch_size,
    train_transform,
    val_transform,
    num_workers=4,
    pin_memory=True,
):
    train_ds = InSARDataset(
        root_dir=TRAIN_DIR,
        transform=train_transform,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_ds = InSARDataset(
        root_dir=VAL_DIR,
        transform=val_transform,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).unsqueeze(1)

            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)

    print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}%")
    print(f"Dice score: {dice_score/len(loader)}")
    model.train()

def save_predictions_as_imgs(loader, model, folder="saved_images/", device="cuda"):
    model.eval()
    if not os.path.exists(folder):
        os.makedirs(folder)
        
    for idx, (x, y) in enumerate(loader):
        x = x.to(device=device)
        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
        
        # Save Prediction
        torchvision.utils.save_image(
            preds, f"{folder}/pred_{idx}.png"
        )
        # Save Ground Truth
        torchvision.utils.save_image(y.float().unsqueeze(1), f"{folder}/true_{idx}.png")
        
        
        torchvision.utils.save_image(x[:, 6:7, :, :], f"{folder}/input_amp_{idx}.png")

    model.train()

In [50]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().to(device=DEVICE).unsqueeze(1)

        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(loss=loss.item())

def main():
    # --- CONFIGURATION ---
    # Point these to your new folders containing the _amp.png, _coh.png, _mask.png files
    # Make sure you have physically moved the files into 'train' and 'val' subfolders!

    # --- TRANSFORMS ---
    train_transform = A.Compose(
        [
            A.RandomRotate90(p=0.75),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.Normalize(
                mean=[0.0] * IN_CHANNELS,  # 18 Channels
                std=[1.0] * IN_CHANNELS,   # 18 Channels
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )

    val_transform = A.Compose(
        [
            A.Normalize(
                mean=[0.0] * IN_CHANNELS,  # 18 Channels
                std=[1.0] * IN_CHANNELS,   # 18 Channels
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )

    # --- DATASET & LOADERS ---
    # 1. Create Train Dataset
    train_ds = InSARDataset(
        root_dir=TRAIN_DIR,
        transform=train_transform,
    )
    
    # 2. Create Validation Dataset (This was missing in your error!)
    val_ds = InSARDataset(
        root_dir=VAL_DIR,
        transform=val_transform,
    )

    # 3. Create Loaders
    train_loader = DataLoader(
        train_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY
    )

    # --- MODEL SETUP ---
    # Change in_channels=18 (All your input channels)
    model = UNET(in_channels=IN_CHANNELS, out_channels=1).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    if LOAD_MODEL:
        load_checkpoint(torch.load("my_checkpoint.pth.tar"), model)

    scaler = torch.cuda.amp.GradScaler()

    # --- TRAINING LOOP ---
    for epoch in range(NUM_EPOCHS):
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
        save_checkpoint(checkpoint)
        
        check_accuracy(val_loader, model, device=DEVICE)
        
        # Save example images
        save_predictions_as_imgs(
            val_loader, model, folder="saved_images/", device=DEVICE
        )

if __name__ == "__main__":
    main()

/tmp/ipykernel_260611/3650552227.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch [1/200]


  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_260611/3650552227.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 1/1 [00:00<00:00,  1.20it/s, loss=0.659]


=> Saving checkpoint
Got 204990/303372 with acc 67.57%
Dice score: 0.0
Epoch [2/200]


100%|██████████| 1/1 [00:00<00:00,  1.19it/s, loss=0.637]


...

=> Saving checkpoint
Got 272559/303372 with acc 89.84%
Dice score: 0.8237333297729492
Epoch [199/200]


100%|██████████| 1/1 [00:00<00:00,  1.19it/s, loss=0.127]


=> Saving checkpoint
Got 272624/303372 with acc 89.86%
Dice score: 0.8228393793106079
Epoch [200/200]


100%|██████████| 1/1 [00:00<00:00,  1.16it/s, loss=0.125]


=> Saving checkpoint
Got 271101/303372 with acc 89.36%
Dice score: 0.8111669421195984
